# Limpeza e Engenharia de Features

Aqui aplico as decisões de limpeza que descrevi no notebook anterior e construo as três bases que vou usar nos modelos: a série de ocupação de leitos, os indicadores por hospital e a matriz de features de risco de readmissão. Centralizei essa lógica em `src/data/load_data.py` e `src/features/engenharia_features.py` para não repetir código em cada notebook — se eu precisar ajustar uma regra de limpeza, mudo em um único lugar.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from src.data.load_data import pipeline_completo
from src.features.engenharia_features import (
    calcular_ocupacao_diaria, calcular_serie_rede,
    calcular_indicadores_hospital, preparar_features_readmissao
)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Pipeline de limpeza

`pipeline_completo()` carrega os dois arquivos brutos e aplica as validações: datas consistentes, permanência entre 1 e 90 dias, idade entre 0 e 110 anos, valor positivo, hospital cadastrado, sem duplicatas. Como vi na EDA que o dataset simulado já nasce limpo, não espero perder muitas linhas — mas rodar essas validações é o que garante que o pipeline não quebra silenciosamente no dia em que eu trocar pelo SIH real, que certamente vai trazer inconsistências.

In [ ]:
internacoes, hospitais = pipeline_completo()
print(f'{len(internacoes):,} internações após limpeza, {len(hospitais)} hospitais')

## 2. Ocupação diária de leitos

Calculo quantos leitos cada hospital tem ocupados em cada dia — a lógica de eventos de entrada (+1) e saída (-1) está em `calcular_ocupacao_diaria()` — e depois agrego numa série única da rede. Corto o início e o fim da série: ver o motivo em `recortar_janela_valida()`, que evita o período de aquecimento do começo (ainda sem pacientes acumulados) e a queda artificial do fim (últimos pacientes recebendo alta sem ninguém novo entrando depois do corte da extração). Nenhum dos dois efeitos é demanda real, só limite dos dados.

In [ ]:
ocupacao = calcular_ocupacao_diaria(internacoes, hospitais)
serie_rede = calcular_serie_rede(ocupacao, data_limite=internacoes['data_internacao'].max())

print(f"Série da rede: {len(serie_rede)} dias, de {serie_rede['data'].min().date()} a {serie_rede['data'].max().date()}")
serie_rede['taxa_ocupacao_rede'].describe()

In [ ]:
fig, ax = plt.subplots()
ax.plot(serie_rede['data'], serie_rede['taxa_ocupacao_rede'], linewidth=1, color='steelblue')
media_movel = serie_rede['taxa_ocupacao_rede'].rolling(30).mean()
ax.plot(serie_rede['data'], media_movel, linewidth=2, color='crimson', label='Média móvel 30 dias')
ax.set_title('Taxa de ocupação da rede hospitalar')
ax.set_xlabel('Data')
ax.set_ylabel('Taxa de ocupação')
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/ocupacao_rede.png', dpi=150)
plt.show()

A rede opera com ocupação média acima de 90%, com picos que passam de 100% — ou seja, mais pacientes internados do que a capacidade nominal calculada para aquele hospital. Isso é consistente com o problema descrito no escopo do projeto: rede sob pressão constante, mais evidente nos hospitais públicos, que calibrei com meta de ocupação mais alta já na etapa de geração dos dados.

## 3. Indicadores por hospital

Consolido os cinco indicadores citados no escopo do projeto — tempo médio de permanência, taxa de readmissão, taxa de ocupação, rotatividade de leitos e taxa de mortalidade — num indicador por hospital. É essa tabela que vai alimentar a clusterização no notebook 05.

In [ ]:
indicadores_hospital = calcular_indicadores_hospital(
    internacoes, ocupacao, hospitais, data_limite=internacoes['data_internacao'].max()
)
indicadores_hospital.sort_values('taxa_ocupacao_media', ascending=False)[
    ['hospital_id', 'tipo_gestao', 'tempo_medio_permanencia', 'taxa_readmissao', 'taxa_ocupacao_media', 'rotatividade_leitos']
]

## 4. Features para o modelo de risco de readmissão

Monto a matriz de features para o classificador — idade, especialidade, diagnóstico, caráter da internação, tempo de permanência, valor e histórico de internações anteriores. Deixo de fora quem morreu durante a internação, porque paciente que não recebeu alta viva não pode ser readmitido — incluir esses casos como "não readmitiu" estaria injetando um viés errado no modelo.

In [ ]:
features_readmissao = preparar_features_readmissao(internacoes)
print(f'{len(features_readmissao):,} internações com alta viva')
print(f"Taxa de readmissão em 30 dias: {features_readmissao['readmissao_30d'].mean()*100:.1f}%")
features_readmissao.head()

## 5. Salvando os dados processados

Salvo as quatro bases em parquet para os próximos notebooks não precisarem recalcular tudo de novo a cada execução.

In [ ]:
PROCESSED = Path('../data/processed')

serie_rede.to_parquet(PROCESSED / 'serie_rede.parquet', index=False)
ocupacao.to_parquet(PROCESSED / 'ocupacao_diaria.parquet', index=False)
indicadores_hospital.to_parquet(PROCESSED / 'indicadores_hospital.parquet', index=False)
features_readmissao.to_parquet(PROCESSED / 'features_readmissao.parquet', index=False)

print('Bases processadas salvas em data/processed/')

## Conclusões

Com a limpeza e as features prontas, sigo para três frentes de modelagem nos próximos notebooks: previsão de ocupação de leitos (03), classificação de risco de readmissão (04) e clusterização de hospitais por perfil operacional (05).